## 1) setup and imports:

In [ ]:
import os, re, json, time, random, hashlib, shelve
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix

# tighter printing
pd.set_option("display.max_colwidth", 140)

## 2) config + robust LLM client (with cache):

In [ ]:
# ---- config (.env optional) ----
API_KEY  = os.getenv("LLM_API_KEY")
API_BASE = os.getenv("LLM_API_BASE", "")
MODEL    = os.getenv("LLM_MODEL",    "meta-llama/Llama-3.3-70B-Instruct")

assert API_BASE.startswith("http"), "Set LLM_API_BASE to your server, e.g. http://host:port"
headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

SESSION = requests.Session()
CACHE_PATH = "llm_cache.db"  # local persistent cache

def chat_text(prompt, model=MODEL, temperature=0.0, max_tokens=8, timeout=60, stop=None, retries=4):
    """Minimal chat client with retries. Returns raw text."""
    url = f"{API_BASE}/api/chat/completions"
    data = {"model": model, "messages":[{"role":"user","content":prompt}],
            "temperature": float(temperature), "max_tokens": int(max_tokens)}
    if stop is not None: data["stop"] = stop

    last = None
    for i in range(retries):
        try:
            r = SESSION.post(url, headers=headers, json=data, timeout=timeout)
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"].strip()
        except requests.exceptions.RequestException as e:
            last = e
            time.sleep(min(6, 0.6*(2**i)) + random.random()*0.2)
    raise RuntimeError(f"LLM call failed after {retries} tries: {last}")

def _cache_key(prompt, **kw):
    payload = {"prompt": prompt, **kw}
    return hashlib.sha1(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()

def chat_text_cached(prompt, **kw):
    key = _cache_key(prompt, **kw)
    with shelve.open(CACHE_PATH) as db:
        if key in db:
            return db[key]
        out = chat_text(prompt, **kw)
        db[key] = out
        return out

## 3) load & prep data:

In [ ]:
import pandas as pd

DATA_PATH = "../data/group_23_url_1_transcript.csv"

# Try reading as semicolon-delimited
df_raw = pd.read_csv(DATA_PATH, sep=";", engine="python")

print(df_raw.columns)

# Now select and rename
df = df_raw.rename(columns={"Emotion_core": "Emotion"})[["Sentence", "Emotion"]].copy()

# Clean whitespace
df["Sentence"] = df["Sentence"].astype(str).str.strip()
df["Emotion"]  = df["Emotion"].astype(str).str.lower().str.strip()

print(df.head())

## 4) stratified split:

In [ ]:
n = len(df)
start = int(n * 0.4)
end = int(n * 0.6)
df_test = df.iloc[start:end]
df_dev = pd.concat([df.iloc[:start], df.iloc[end:]]).reset_index(drop=True)

In [ ]:
# === Evaluate helper (robust) ===
ALLOWED = ["happiness","sadness","anger","surprise","fear","disgust","neutral"]

from sklearn.metrics import classification_report, f1_score, confusion_matrix

def evaluate(name, df_subset, predict_fn, show_cm=False, show_norm_cm=True, return_metrics=False):
    """
    Evaluate a predict_fn(text)->label or raw response.
    - Coerces to ALLOWED labels if possible.
    - Prints macro/weighted/micro F1, neutral rates, and (optionally) confusion matrices.
    """
    # Fallback normalizer if parse_label_plain isn't available
    import re, json
    _SYN = {
        "happy":"happiness","joy":"happiness","joyful":"happiness","glad":"happiness","excited":"happiness",
        "sad":"sadness","sorrow":"sadness","unhappy":"sadness",
        "angry":"anger","mad":"anger","furious":"anger","annoyed":"anger",
        "surprised":"surprise","shocked":"surprise","astonished":"surprise","shock":"surprise",
        "afraid":"fear","scared":"fear","anxious":"fear","worry":"fear","worried":"fear",
        "disgusted":"disgust","gross":"disgust","yuck":"disgust",
        "none":"neutral","objective":"neutral","no emotion":"neutral",
    }
    def _normalize_label(raw):
        if not isinstance(raw, str) or not raw.strip():
            return "neutral"
        s = raw.strip().lower()
        # JSON path
        try:
            jd = json.loads(s)
            if isinstance(jd, dict) and "label" in jd:
                s = str(jd["label"]).strip().lower()
        except Exception:
            pass
        m = re.search(r'\b(happiness|sadness|anger|surprise|fear|disgust|neutral)\b', s)
        if m: return m.group(1)
        for k,v in _SYN.items():
            if re.search(rf'\b{k}\b', s): return v
        if "sad" in s: return "sadness"
        return "neutral"

    def _coerce(x):
        try:
            from types import FunctionType
            # use user's parser if present
            if 'parse_label_plain' in globals() and callable(globals()['parse_label_plain']):
                return globals()['parse_label_plain'](x)
        except Exception:
            pass
        return _normalize_label(x)

    y_true = df_subset["Emotion"].astype(str).str.lower().tolist()
    y_pred = []
    for txt in df_subset["Sentence"].tolist():
        try:
            out = predict_fn(txt)
        except Exception:
            out = "neutral"
        y_pred.append(_coerce(out))

    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred, digits=3, zero_division=0, labels=ALLOWED))

    macro = f1_score(y_true, y_pred, average="macro", labels=ALLOWED, zero_division=0)
    weighted = f1_score(y_true, y_pred, average="weighted", labels=ALLOWED, zero_division=0)
    micro = f1_score(y_true, y_pred, average="micro", labels=ALLOWED, zero_division=0)
    print(f"Macro-F1: {macro:.3f} | Weighted-F1: {weighted:.3f} | Micro-F1: {micro:.3f}")

    # Neutral rate sanity
    import pandas as _pd, numpy as _np
    true_neutral = (_pd.Series(y_true) == "neutral").mean()
    pred_neutral = (_pd.Series(y_pred) == "neutral").mean()
    print(f"Neutral rate — True: {true_neutral:.2%} | Pred: {pred_neutral:.2%}")

    # Confusion matrices
    cm = confusion_matrix(y_true, y_pred, labels=ALLOWED)
    if show_cm:
        import pandas as pd
        cm_df = pd.DataFrame(cm, index=[f"T:{l}" for l in ALLOWED], columns=[f"P:{l}" for l in ALLOWED])
        print("\nConfusion Matrix (counts):\n", cm_df)
    if show_norm_cm:
        import numpy as np, pandas as pd
        with np.errstate(all='ignore'):
            row_sums = cm.sum(axis=1, keepdims=True)
            norm = cm / np.where(row_sums == 0, 1, row_sums)
        norm_df = pd.DataFrame(np.round(norm, 3), index=[f"T:{l}" for l in ALLOWED], columns=[f"P:{l}" for l in ALLOWED])
        print("\nConfusion Matrix (row-normalized):\n", norm_df)

    if return_metrics:
        return y_pred, {
            "macro_f1": float(macro),
            "weighted_f1": float(weighted),
            "micro_f1": float(micro),
            "neutral_rate_true": float(true_neutral),
            "neutral_rate_pred": float(pred_neutral),
            "confusion_matrix": cm.tolist()
        }
    return y_pred

In [ ]:
# ==== Constants & parsers ====
ALLOWED = ["happiness","sadness","anger","surprise","fear","disgust","neutral"]
NON_NEUTRAL = ["happiness","sadness","anger","surprise","fear","disgust"]

_SYNONYMS = {
    "happy":"happiness","joy":"happiness","joyful":"happiness","glad":"happiness","excited":"happiness",
    "sad":"sadness","sorrow":"sadness","unhappy":"sadness",
    "angry":"anger","mad":"anger","furious":"anger","annoyed":"anger",
    "surprised":"surprise","shocked":"surprise","astonished":"surprise","shock":"surprise",
    "afraid":"fear","scared":"fear","anxious":"fear","worry":"fear","worried":"fear",
    "disgusted":"disgust","gross":"disgust","yuck":"disgust",
    "none":"neutral","objective":"neutral","no emotion":"neutral",
}

def _normalize_label(raw: str) -> str:
    if not isinstance(raw, str) or not raw.strip():
        return "neutral"
    s = raw.strip().lower()

    # Try JSON: {"label": "...", "probs": {...}}
    try:
        jd = json.loads(s)
        if isinstance(jd, dict) and "label" in jd:
            s = str(jd["label"]).strip().lower()
    except Exception:
        pass

    m = re.search(r'\b(happiness|sadness|anger|surprise|fear|disgust|neutral)\b', s)
    if m:
        return m.group(1)
    for k,v in _SYNONYMS.items():
        if re.search(rf'\b{k}\b', s):
            return v
    if "sad" in s:
        return "sadness"
    return "neutral"

def parse_label_plain(s: str) -> str:
    return _normalize_label(s)

def parse_json_probs(s: str):
    """Return (label, probs_dict over ALLOWED). If parsing fails: normalized label + uniform probs."""
    try:
        jd = json.loads(s)
        lab = _normalize_label(jd.get("label","neutral"))
        probs = jd.get("probs", {})
        p = {k: float(probs.get(k, 0.0)) for k in ALLOWED}
        tot = sum(p.values())
        if tot <= 0:
            p = {k: 1.0/len(ALLOWED) for k in ALLOWED}
        else:
            p = {k: v/tot for k,v in p.items()}
        return lab, p
    except Exception:
        return _normalize_label(s), {k: 1.0/len(ALLOWED) for k in ALLOWED}

# ==== Priors and aggregation ====
def class_priors_from_dev(df_dev, neutral_soften=0.85, power=0.5):
    vc = df_dev["Emotion"].value_counts().reindex(ALLOWED).fillna(1)
    inv = 1.0 / np.power(vc.values, power)   # inverse-sqrt-ish
    pri = inv / inv.sum()
    pri = dict(zip(ALLOWED, pri))
    pri["neutral"] *= neutral_soften
    s = sum(pri.values())
    for k in pri:
        pri[k] /= s
    return pri

def combine_probs_with_priors(prob_list, priors, gamma=0.6, neutral_demotion=0.9):
    """
    prob_list: list of dicts over ALLOWED
    priors: dict over ALLOWED
    gamma in [0..1]: weight of priors in the mixture
    neutral_demotion < 1: reduces neutral score to curb overuse
    """
    agg = {k:0.0 for k in ALLOWED}
    for probs in prob_list:
        for lab in ALLOWED:
            agg[lab] += (1-gamma)*probs.get(lab,0.0) + gamma*priors.get(lab, 1.0/len(ALLOWED))
    agg["neutral"] *= float(neutral_demotion)
    s = sum(agg.values()) or 1.0
    agg = {k: v/s for k,v in agg.items()}
    label = max(agg, key=agg.get)
    return label, agg

In [ ]:
# If you already defined these earlier, you can skip or keep this as an overwrite.

FILLER_RE = re.compile(r'^(yeah|yep|ok(ay)?|mm+|uh+|oh)\.?$', re.I)
LEX = {
    "disgust":   re.compile(r"\b(disgust(ing)?|gross|feel sick|nausea|yuck|ew+w?)\b", re.I),
    "happiness": re.compile(r"\b(thanks|thank you|glad|happy|proud|win|victor(y|ies)|congrats?)\b", re.I),
    "sadness":   re.compile(r"\b(breaks my heart|farewell|gutted|sad|goodbye|loss|heartbroken|devastated)\b", re.I),
    "anger":     re.compile(r"\b(god ?damn|wrong one|angry|furious|frustrat(ed|ing)|mad)\b|^go!?$", re.I),
    "surprise":  re.compile(r"\b(didn'?t expect|what just happened|twist|shocked|wow|no way|ready\?)\b", re.I),
    "fear":      re.compile(r"\b(dangerous|target me|worried|afraid|scared|terrified|threat|unsafe|nervous|anxious)\b", re.I),
}

def is_trivial_neutral(t): 
    t = (t or "").strip()
    return len(t) < 4 or FILLER_RE.match(t) is not None

def rule_label(text):
    if is_trivial_neutral(text): 
        return "neutral"
    if LEX["disgust"].search(text or ""):  # rare class first
        return "disgust"
    for lab, rx in LEX.items():
        if lab == "disgust":
            continue
        if rx.search(text or ""):
            return lab
    return None

def has_strong_emotion_cue(text):
    if re.search(r"[!?\u2757]", text): 
        return True
    if re.search(r"\b(i('| )?m|i am|i feel|it makes me)\b", text, re.I):
        return True
    return any(rx.search(text or "") for rx in LEX.values())

def post_nudge(label, text):
    t = (text or "").lower()
    if label == "surprise" and any(k in t for k in ["challenge", "immunity", "survivors ready", "go!"]):
        return "surprise"
    if label == "happiness" and any(k in t for k in ["thanks", "glad", "proud", "we did it", "so happy"]):
        return "happiness"
    return label

In [ ]:
CURATED_FEWSHOT = [
    ('We finally did it!', 'happiness'),
    ('I miss her so much.', 'sadness'),
    ('God damn it!', 'anger'),
    ('Wait, what just happened?', 'surprise'),
    ("I don't feel safe about this.", 'fear'),
    ("That's disgusting.", 'disgust'),
    ('Six are left.', 'neutral'),
]

DEFINITIONS = """Definitions:
- happiness: positive affect/relief/success (e.g., we did it, proud, happy, love).
- sadness: loss/sorrow/regret (e.g., miss, gutted, heartbroken).
- anger: irritation/hostility/injustice (e.g., pissed, unfair, furious).
- surprise: unexpectedness/shock (valence ±), (e.g., what?, can't believe, twist, immunity shock).
- fear: threat/anxiety/worry (e.g., scared, target me, unsafe, nervous).
- disgust: aversion/moral revulsion (e.g., disgusting, gross, sick).
- neutral: purely factual/structural (e.g., 'Survivors ready?', 'Six are left.')."""

RULES = """Rules:
- Use 'neutral' ONLY if there is no affective cue. Exclamations, appraisals, or emotion words imply a non-neutral label.
- Output must select exactly one label from: [happiness, sadness, anger, surprise, fear, disgust, neutral].
"""

def fewshot_block(pairs=CURATED_FEWSHOT):
    rows = [f"\"{s}\" -> {l}" for s,l in pairs]
    return "Few-shot (Sentence -> Label):\n" + "\n".join(rows)

def combo_json_prompt(text, fewshot_pairs=CURATED_FEWSHOT):
    return f"""{DEFINITIONS}

{RULES}

{fewshot_block(fewshot_pairs)}

Now classify the new sentence and respond ONLY with strict JSON:
{{"label": "<one_of_allowed>", "probs": {{"happiness":x,"sadness":x,"anger":x,"surprise":x,"fear":x,"disgust":x,"neutral":x}}}}
The probabilities must be non-negative and sum to 1.

Sentence: "{text}"
"""

def combo_label_prompt_no_neutral(text, fewshot_pairs=CURATED_FEWSHOT):
    return f"""{DEFINITIONS}

Rules (no-neutral):
- You MUST choose only from: [happiness, sadness, anger, surprise, fear, disgust].
- Do not output 'neutral' under any circumstance.

{fewshot_block(fewshot_pairs)}

Answer with ONE label only.

Sentence: "{text}"
"""

In [ ]:
def _ask_json(prompt, temperature=0.0, max_tokens=180):
    raw = chat_text_cached(prompt, temperature=temperature, max_tokens=max_tokens)
    lab, probs = parse_json_probs(raw)
    return lab, probs, raw

def _ask_label(prompt, temperature=0.2, max_tokens=16):
    raw = chat_text_cached(prompt, temperature=temperature, max_tokens=max_tokens)
    lab = parse_label_plain(raw)
    return lab, raw

In [ ]:
def happy_surprise_prompt(text):
    return f"""Disambiguate ONLY between 'happiness' and 'surprise'.

Guidance:
- If the sentence expresses unexpectedness/shock (e.g., "what", "can't believe", "twist", "immunity"), choose 'surprise'.
- If it expresses positive achievement/joy/relief (e.g., "we did it", "so happy", "proud", "congrats"), choose 'happiness'.

Answer with ONE word only: happiness or surprise.

Sentence: "{text}"
"""

def resolve_happy_surprise(text):
    lab, raw = _ask_label(happy_surprise_prompt(text), temperature=0.0, max_tokens=8)
    return lab if lab in ("happiness","surprise") else None

In [ ]:
def classify_two_stage_combo(
    text,
    priors,
    fewshot_pairs=CURATED_FEWSHOT,
    t_stage1=0.0,
    t_stage2_list=(0.0, 0.2, 0.35),
    low_conf_thresh=0.45,
    neutral_demotion=0.88,
    prior_gamma=0.6
):
    # A) High-precision rules first
    rb = rule_label(text)
    if rb:
        return rb

    # B) Stage 1 — JSON probs (neutral allowed)
    lab1, probs1, raw1 = _ask_json(combo_json_prompt(text, fewshot_pairs), temperature=t_stage1, max_tokens=220)

    prob_list = [probs1]
    final_label = lab1

    # C) Escalate if neutral / low confidence / strong cues
    if lab1 == "neutral" or probs1.get(lab1, 0.0) < low_conf_thresh or has_strong_emotion_cue(text):
        # No-neutral ballots
        for t in t_stage2_list:
            lab2, raw2 = _ask_label(combo_label_prompt_no_neutral(text, fewshot_pairs), temperature=t, max_tokens=12)
            if lab2 in NON_NEUTRAL:
                peaked = {k: 1e-6 for k in ALLOWED}
                peaked[lab2] = 1.0
                prob_list.append(peaked)

        # Targeted happiness↔surprise resolver if ambiguous
        if max(probs1.get("happiness",0), probs1.get("surprise",0)) < 0.65:
            hs = resolve_happy_surprise(text)
            if hs in ("happiness","surprise"):
                peaked = {k: 1e-6 for k in ALLOWED}; peaked[hs] = 1.0
                prob_list.append(peaked)

        # Aggregate with priors and neutral demotion
        final_label, final_probs = combine_probs_with_priors(prob_list, priors, gamma=prior_gamma, neutral_demotion=neutral_demotion)
    else:
        # confident Stage-1; still apply mild smoothing
        final_label, final_probs = combine_probs_with_priors([probs1], priors, gamma=prior_gamma, neutral_demotion=neutral_demotion)

    # D) Survivor-specific post nudge
    final_label = post_nudge(final_label, text)
    return final_label

In [ ]:
# Compute priors from DEV
priors = class_priors_from_dev(df_dev, neutral_soften=0.85, power=0.5)
priors

def predict_combo_plain(text):
    return classify_two_stage_combo(
        text=text,
        priors=priors,
        fewshot_pairs=CURATED_FEWSHOT,
        t_stage1=0.0,
        t_stage2_list=(0.0, 0.2),
        low_conf_thresh=0.45,
        neutral_demotion=0.90,
        prior_gamma=0.55,
    )

def predict_combo_strong(text):
    return classify_two_stage_combo(
        text=text,
        priors=priors,
        fewshot_pairs=CURATED_FEWSHOT,
        t_stage1=0.0,
        t_stage2_list=(0.0, 0.2, 0.35, 0.5),
        low_conf_thresh=0.50,
        neutral_demotion=0.85,
        prior_gamma=0.65,
    )

In [ ]:
# Compute priors from DEV
priors = class_priors_from_dev(df_dev, neutral_soften=0.85, power=0.5)
priors

def predict_combo_plain(text):
    return classify_two_stage_combo(
        text=text,
        priors=priors,
        fewshot_pairs=CURATED_FEWSHOT,
        t_stage1=0.0,
        t_stage2_list=(0.0, 0.2),
        low_conf_thresh=0.45,
        neutral_demotion=0.90,
        prior_gamma=0.55,
    )

def predict_combo_strong(text):
    return classify_two_stage_combo(
        text=text,
        priors=priors,
        fewshot_pairs=CURATED_FEWSHOT,
        t_stage1=0.0,
        t_stage2_list=(0.0, 0.2, 0.35, 0.5),
        low_conf_thresh=0.50,
        neutral_demotion=0.85,
        prior_gamma=0.65,
    )

In [ ]:
# === Baseline two-pass (plain) ===
# Uses your chat_text_cached and provides local fallbacks for any missing helpers.

# 1) Prompts
PLAIN_PROMPT = """Pick exactly ONE label from [happiness, sadness, anger, surprise, fear, disgust, neutral].
If you are unsure between an emotional label and neutral, prefer the emotional label.
Sentence: {text}
Answer with the single label only."""
PLAIN_PROMPT_NO_NEUTRAL = """Pick exactly ONE label from [happiness, sadness, anger, surprise, fear, disgust].
Assume the sentence is not neutral; pick the closest label.
Sentence: {text}
Answer with the single label only."""

# 2) Tiny ask wrapper (uses your cache)
def _ask(prompt):
    return chat_text_cached(prompt, temperature=0.0, max_tokens=12)

# 3) Happiness↔Surprise patch
SURPRISE_CUES = re.compile(r"\b(ready\?|get to today'?s immunity|challenge|immunity|survivors ready\?)", re.I)
HAPPY_CUES    = re.compile(r"\b(thanks|congrats|we did it|nice to meet you|so happy|proud)\b", re.I)

def happiness_surprise_patch(pred, text):
    if pred == "happiness" and SURPRISE_CUES.search(text or ""):
        return "surprise"
    if pred == "surprise" and HAPPY_CUES.search(text or ""):
        return "happiness"
    return pred

# 4) Optional fallbacks for rules/cues if you didn't define them earlier
if 'rule_label' not in globals():
    FILLER_RE = re.compile(r'^(yeah|yep|ok(ay)?|mm+|uh+|oh)\.?$', re.I)
    def is_trivial_neutral(t): 
        t = (t or "").strip()
        return len(t) < 4 or FILLER_RE.match(t) is not None
    def rule_label(text):
        if is_trivial_neutral(text): 
            return "neutral"
        return None

if 'has_strong_emotion_cue' not in globals():
    def has_strong_emotion_cue(text):
        # basic fallback (punctuation or obvious emotion terms)
        return bool(re.search(r"[!?\u2757]", text or "")) or bool(
            re.search(r"\b(happy|sad|angry|furious|shocked|surprised|afraid|scared|worried|disgust)\b", (text or "").lower())
        )

# 5) Parser fallback if missing
if 'parse_label_plain' not in globals():
    def parse_label_plain(s: str):
        s = str(s).lower()
        m = re.search(r'\b(happiness|sad(ness)?|anger|surprise|fear|disgust|neutral)\b', s)
        if not m: 
            return "neutral"
        lab = m.group(1)
        return "sadness" if lab.startswith("sad") else lab

# 6) The classifier
def classify_two_pass_plain(text: str):
    # hard rules first
    rb = rule_label(text)
    if rb:
        return rb

    # pass 1 — neutral allowed
    raw1 = _ask(PLAIN_PROMPT.format(text=text))
    lab1 = parse_label_plain(raw1)
    if lab1 in ALLOWED and lab1 != "neutral":
        return happiness_surprise_patch(lab1, text)

    # if neutral or unparsed and *no cues* → neutral
    if (lab1 is None or lab1 == "neutral") and not has_strong_emotion_cue(text):
        return "neutral"

    # pass 2 — forbid neutral
    raw2 = _ask(PLAIN_PROMPT_NO_NEUTRAL.format(text=text))
    lab2 = parse_label_plain(raw2)
    lab2 = happiness_surprise_patch(lab2, text) if lab2 else None
    return lab2 if lab2 in ALLOWED and lab2 != "neutral" else "neutral"

In [ ]:
_ = evaluate("Combo JSON+FewShot+Rules (strong)", df_dev, predict_combo_strong, show_cm=True)

In [ ]:
_ = evaluate("Two-pass PLAIN (old baseline)", df_dev, classify_two_pass_plain, show_cm=False)
_ = evaluate("Combo JSON+FewShot+Rules (plain)", df_dev, predict_combo_plain, show_cm=False)

In [ ]:
def rebalance_neutral_combo(df_subset, preds, cap=0.60, reclassify_fn=predict_combo_strong, max_flips=None):
    preds = np.array(preds, dtype=object)
    mask = (preds == "neutral")
    rate = mask.mean()
    if rate <= cap:
        return preds

    idxs = np.where(mask)[0]
    # prioritize neutrals that actually have strong cues
    cue_flags = df_subset.iloc[idxs]["Sentence"].apply(has_strong_emotion_cue).to_numpy()
    order = np.r_[np.where(cue_flags)[0], np.where(~cue_flags)[0]]

    need = int(round((rate - cap) * len(preds)))
    if max_flips is not None:
        need = min(need, max_flips)

    take = idxs[order[:need]]

    for i in take:
        text = df_subset.iloc[i]["Sentence"]
        new_lab = reclassify_fn(text)
        preds[i] = post_nudge(new_lab, text)

    return preds

def evaluate_combo_with_cap(name, df_subset, base_fn, cap=0.60, reclassify_fn=predict_combo_strong, max_flips=None):
    y_true = df_subset["Emotion"].to_numpy()
    base_preds = df_subset["Sentence"].apply(base_fn).to_numpy()
    adj_preds = rebalance_neutral_combo(df_subset, base_preds, cap=cap, reclassify_fn=reclassify_fn, max_flips=max_flips)
    print(f"\n=== {name} (neutral cap {cap:.0%}, max_flips={max_flips}) ===")
    print(classification_report(y_true, adj_preds, digits=3, zero_division=0, labels=ALLOWED))
    macro = f1_score(y_true, adj_preds, average="macro", labels=ALLOWED)
    weighted = f1_score(y_true, adj_preds, average="weighted", labels=ALLOWED)
    print(f"Macro-F1: {macro:.3f} | Weighted-F1: {weighted:.3f}")
    return adj_preds

# Try a cap of 60%
_ = evaluate_combo_with_cap("Combo STRONG + neutral-cap", df_dev, predict_combo_strong, cap=0.60, reclassify_fn=predict_combo_strong, max_flips=None)

In [ ]:
print("\n--- TEST ---")
_ = evaluate("Combo JSON+FewShot+Rules (strong)", df_test, predict_combo_strong, show_cm=True)
_ = evaluate_combo_with_cap("Combo STRONG + neutral-cap", df_test, predict_combo_strong, cap=0.60, reclassify_fn=predict_combo_strong, max_flips=None)

## 2nd iteration:

In [ ]:
# ==== Iteration 2: Setup ====
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    ALLOWED
except NameError:
    ALLOWED = ["happiness","sadness","anger","surprise","fear","disgust","neutral"]

assert "df_dev" in globals() and "df_test" in globals(), "Run your data load & stratified split first."
assert "chat_text_cached" in globals(), "Your cached LLM client (chat_text_cached) must be defined earlier."
assert "parse_label_plain" in globals(), "Make sure parse_label_plain exists (from your earlier helpers)."
assert "parse_json_probs" in globals(), "Make sure parse_json_probs exists (JSON label+probs to dict)."
assert "rule_label" in globals() and "has_strong_emotion_cue" in globals() and "post_nudge" in globals(), \
    "Need rule_label, has_strong_emotion_cue, post_nudge from your earlier blocks."

In [ ]:
# --- Retrieval pool (cap neutrals & ensure rare classes appear) ---
def build_pool(df, per_class_caps=None):
    if per_class_caps is None:
        per_class_caps = {"neutral": 80, "happiness": 60, "fear": 60, "sadness": 60, "anger": 50, "surprise": 50, "disgust": 20}
    frames = []
    for lab in ALLOWED:
        cap = per_class_caps.get(lab, 50)
        frames.append(df[df["Emotion"]==lab].head(cap))
    pool = pd.concat(frames).reset_index(drop=True)
    return pool

pool_iter2 = build_pool(df_dev)

# --- TF-IDF index over pool sentences ---
tfidf_iter2 = TfidfVectorizer(max_features=20000, ngram_range=(1,2), lowercase=True, min_df=1)
X_pool_iter2 = tfidf_iter2.fit_transform(pool_iter2["Sentence"].tolist())

def retrieve_examples_iter2(query_text, k=6, per_label_max=2):
    if not isinstance(query_text, str) or not query_text.strip():
        return []
    qv = tfidf_iter2.transform([query_text])
    sims = cosine_similarity(qv, X_pool_iter2)[0]
    idx = np.argsort(-sims)
    shots, per_lab = [], {lab:0 for lab in ALLOWED}
    for i in idx:
        row = pool_iter2.iloc[i]
        lab = row["Emotion"]
        if per_lab[lab] < per_label_max:
            shots.append((row["Sentence"], lab))
            per_lab[lab] += 1
        if len(shots) >= k:
            break
    # Ensure at least one neutral anchor if none selected
    if not any(l=="neutral" for _, l in shots):
        neut = pool_iter2[pool_iter2["Emotion"]=="neutral"]
        if not neut.empty:
            shots.append((neut.iloc[0]["Sentence"], "neutral"))
    return shots[:k]

In [ ]:
DEFS_ITER2 = """Definitions:
- happiness: positive affect/relief/success (we did it, proud, happy, love).
- sadness: loss/sorrow/regret (miss, gutted, heartbroken).
- anger: irritation/hostility/injustice (pissed, unfair, furious).
- surprise: unexpectedness/shock (valence ±) (what?, can't believe, twist, immunity shock).
- fear: threat/anxiety/worry (scared, target me, unsafe, nervous).
- disgust: aversion/moral revulsion (disgusting, gross, sick).
- neutral: purely factual/structural (e.g., 'Survivors ready?', 'Six are left.')."""

RULES_ITER2 = """Rules:
- Use 'neutral' ONLY when there is no affective cue. Exclamation, appraisal, emotion words, or sarcasm imply a non-neutral label.
- Choose exactly one label from [happiness, sadness, anger, surprise, fear, disgust, neutral].
"""

def fewshot_block_iter2(pairs):
    return "Few-shot (Sentence -> Label):\n" + "\n".join([f"\"{s}\" -> {l}" for s,l in pairs])

def json_prompt_iter2(text, shots):
    return f"""{DEFS_ITER2}

{RULES_ITER2}

{fewshot_block_iter2(shots)}

Respond ONLY with strict JSON:
{{"label":"<one_of_allowed>", "probs":{{"happiness":x,"sadness":x,"anger":x,"surprise":x,"fear":x,"disgust":x,"neutral":x}}}}
Probabilities must be non-negative and sum to 1.

Sentence: "{text}"
"""

def no_neutral_prompt_iter2(text, shots):
    return f"""{DEFS_ITER2}

Rules (no-neutral):
- Choose only from [happiness, sadness, anger, surprise, fear, disgust]. Do NOT output 'neutral'.

{fewshot_block_iter2(shots)}

Answer with ONE label only.

Sentence: "{text}"
"""

In [ ]:
def ask_json_iter2(text, shots, temperature=0.0, max_tokens=220):
    raw = chat_text_cached(json_prompt_iter2(text, shots), temperature=temperature, max_tokens=max_tokens)
    lab, probs = parse_json_probs(raw)
    return lab, probs, raw

def ask_label_no_neutral_iter2(text, shots, temperature=0.2, max_tokens=16):
    raw = chat_text_cached(no_neutral_prompt_iter2(text, shots), temperature=temperature, max_tokens=max_tokens)
    lab = parse_label_plain(raw)
    return lab, raw

In [ ]:
def class_priors_from_dev_iter2(df, neutral_soften=0.85, power=0.5):
    vc = df["Emotion"].value_counts().reindex(ALLOWED).fillna(1)
    inv = 1.0 / np.power(vc.values, power)
    pri = inv / inv.sum()
    pri = dict(zip(ALLOWED, pri))
    pri["neutral"] *= neutral_soften
    s = sum(pri.values())
    for k in pri: pri[k] /= s
    return pri

priors_iter2 = class_priors_from_dev_iter2(df_dev, neutral_soften=0.85, power=0.5)

def combine_probs_with_priors_iter2(prob_list, priors, gamma=0.6, neutral_demotion=0.88):
    agg = {k:0.0 for k in ALLOWED}
    for probs in prob_list:
        for lab in ALLOWED:
            agg[lab] += (1-gamma)*probs.get(lab,0.0) + gamma*priors.get(lab, 1.0/len(ALLOWED))
    agg["neutral"] *= float(neutral_demotion)
    s = sum(agg.values()) or 1.0
    agg = {k: v/s for k,v in agg.items()}
    label = max(agg, key=agg.get)
    return label, agg

def happy_surprise_prompt_iter2(text):
    return f"""Choose ONLY 'happiness' or 'surprise'.
Unexpectedness/shock → 'surprise'. Achievement/joy/relief → 'happiness'.
Answer with one word.
Sentence: "{text}"
"""

def resolve_happy_surprise_iter2(text):
    raw = chat_text_cached(happy_surprise_prompt_iter2(text), temperature=0.0, max_tokens=8)
    lab = parse_label_plain(raw)
    return lab if lab in ("happiness","surprise") else None

In [ ]:
def classify_combo_iter2(
    text,
    t_stage1=0.0,
    t_stage2_list=(0.0, 0.2),   # small diversity
    low_conf_thresh=0.45,
    prior_gamma=0.6,
    neutral_demotion=0.88
):
    # 1) High-precision rules first
    rb = rule_label(text)
    if rb:
        return rb

    # 2) Retrieve episode-specific shots
    shots = retrieve_examples_iter2(text, k=6, per_label_max=2)

    # 3) Stage 1: rubric + JSON probs
    lab1, probs1, _ = ask_json_iter2(text, shots, temperature=t_stage1, max_tokens=220)
    prob_list = [probs1]

    # 4) Confidence/neutral-triggered fallback (no-neutral)
    escalate = (lab1 == "neutral") or (probs1.get(lab1, 0.0) < low_conf_thresh) or has_strong_emotion_cue(text)
    if escalate and t_stage2_list:
        for t in t_stage2_list:
            lab2, _ = ask_label_no_neutral_iter2(text, shots, temperature=t, max_tokens=12)
            if lab2 in ("happiness","sadness","anger","surprise","fear","disgust"):
                peaked = {k: 1e-6 for k in ALLOWED}; peaked[lab2] = 1.0
                prob_list.append(peaked)

    # 5) Targeted happiness↔surprise disambiguation if ambiguous
    if max(probs1.get("happiness",0), probs1.get("surprise",0)) < 0.65:
        hs = resolve_happy_surprise_iter2(text)
        if hs:
            peaked = {k: 1e-6 for k in ALLOWED}; peaked[hs] = 1.0
            prob_list.append(peaked)

    # 6) Aggregate with priors + neutral demotion
    final_label, _ = combine_probs_with_priors_iter2(prob_list, priors_iter2, gamma=prior_gamma, neutral_demotion=neutral_demotion)
    return post_nudge(final_label, text)

In [ ]:
# Collect aggregated probs on DEV (single-pass for calibration simplicity)
def collect_probs_dev_iter2(df_subset):
    probs_all, y_true = [], []
    for s, y in zip(df_subset["Sentence"].tolist(), df_subset["Emotion"].tolist()):
        shots = retrieve_examples_iter2(s, k=6, per_label_max=2)
        _, p1, _ = ask_json_iter2(s, shots, temperature=0.0, max_tokens=220)
        # mild prior mix even during calibration to reflect runtime behavior
        _, agg = combine_probs_with_priors_iter2([p1], priors_iter2, gamma=0.6, neutral_demotion=0.90)
        probs_all.append(agg)
        y_true.append(y)
    return y_true, probs_all

y_dev_iter2, probs_dev_iter2 = collect_probs_dev_iter2(df_dev)

def temperature_scale_iter2(probs: dict, T: float) -> dict:
    eps = 1e-8
    q = {k: (probs[k]+eps)**(1.0/T) for k in probs}
    s = sum(q.values())
    return {k: v/s for k,v in q.items()}

def fit_T_iter2(y_true, probs_list, grid=(0.6,0.8,1.0,1.2,1.5,2.0,2.5,3.0)):
    bestT, bestF = 1.0, -1.0
    for T in grid:
        preds = []
        for p in probs_list:
            q = temperature_scale_iter2(p, T)
            preds.append(max(q, key=q.get))
        f = f1_score(y_true, preds, average="macro", labels=ALLOWED, zero_division=0)
        if f > bestF:
            bestT, bestF = T, f
    return bestT, bestF

bestT_iter2, bestF_iter2 = fit_T_iter2(y_dev_iter2, probs_dev_iter2)
print("Iter2 — Best temperature on DEV:", bestT_iter2, "macro-F1:", round(bestF_iter2, 3))

In [ ]:
def classify_combo_iter2_calibrated(text, bestT=1.0):
    # Same as classify_combo_iter2 but applies T to final aggregated probs before argmax
    rb = rule_label(text)
    if rb: return rb

    shots = retrieve_examples_iter2(text, k=6, per_label_max=2)
    lab1, p1, _ = ask_json_iter2(text, shots, temperature=0.0, max_tokens=220)
    prob_list = [p1]

    escalate = (lab1 == "neutral") or (p1.get(lab1, 0.0) < 0.45) or has_strong_emotion_cue(text)
    if escalate:
        for t in (0.0, 0.2):
            lab2, _ = ask_label_no_neutral_iter2(text, shots, temperature=t, max_tokens=12)
            if lab2 in ("happiness","sadness","anger","surprise","fear","disgust"):
                peaked = {k: 1e-6 for k in ALLOWED}; peaked[lab2] = 1.0
                prob_list.append(peaked)

    if max(p1.get("happiness",0), p1.get("surprise",0)) < 0.65:
        hs = resolve_happy_surprise_iter2(text)
        if hs:
            peaked = {k: 1e-6 for k in ALLOWED}; peaked[hs] = 1.0
            prob_list.append(peaked)

    _, agg = combine_probs_with_priors_iter2(prob_list, priors_iter2, gamma=0.6, neutral_demotion=0.88)
    q = temperature_scale_iter2(agg, bestT)
    return post_nudge(max(q, key=q.get), text)

In [ ]:
# DEV
_ = evaluate("Iter2 — Combo (single-pass+fallback)", df_dev, classify_combo_iter2, show_cm=True)
_ = evaluate("Iter2 — Combo + calibration", df_dev, lambda s: classify_combo_iter2_calibrated(s, bestT_iter2), show_cm=True)

# TEST
print("\n--- TEST ---")
_ = evaluate("Iter2 — Combo + calibration", df_test, lambda s: classify_combo_iter2_calibrated(s, bestT_iter2), show_cm=True)

In [ ]:
def rebalance_neutral_iter2(df_subset, preds, cap=0.60, reclassify_fn=None, max_flips=None):
    if reclassify_fn is None:
        reclassify_fn = lambda s: classify_combo_iter2_calibrated(s, bestT_iter2)
    preds = np.array(preds, dtype=object)
    mask = (preds == "neutral")
    rate = mask.mean()
    if rate <= cap:
        return preds
    idxs = np.where(mask)[0]
    cue_flags = df_subset.iloc[idxs]["Sentence"].apply(has_strong_emotion_cue).to_numpy()
    order = np.r_[np.where(cue_flags)[0], np.where(~cue_flags)[0]]
    need = int(round((rate - cap) * len(preds)))
    if max_flips is not None:
        need = min(need, max_flips)
    take = idxs[order[:need]]
    for i in take:
        txt = df_subset.iloc[i]["Sentence"]
        preds[i] = post_nudge(reclassify_fn(txt), txt)
    return preds

def evaluate_iter2_with_cap(name, df_subset, base_fn, cap=0.60, max_flips=None):
    y_true = df_subset["Emotion"].to_numpy()
    base_preds = df_subset["Sentence"].apply(base_fn).to_numpy()
    adj_preds = rebalance_neutral_iter2(df_subset, base_preds, cap=cap, max_flips=max_flips)
    print(f"\n=== {name} (neutral cap {cap:.0%}, max_flips={max_flips}) ===")
    print(classification_report(y_true, adj_preds, digits=3, zero_division=0, labels=ALLOWED))
    macro = f1_score(y_true, adj_preds, average="macro", labels=ALLOWED, zero_division=0)
    weighted = f1_score(y_true, adj_preds, average="weighted", labels=ALLOWED, zero_division=0)
    print(f"Macro-F1: {macro:.3f} | Weighted-F1: {weighted:.3f}")
    return adj_preds

# Try on DEV, then TEST
_ = evaluate_iter2_with_cap("Iter2 — Combo+calib + neutral-cap", df_dev, lambda s: classify_combo_iter2_calibrated(s, bestT_iter2), cap=0.60, max_flips=None)
_ = evaluate_iter2_with_cap("Iter2 — Combo+calib + neutral-cap", df_test, lambda s: classify_combo_iter2_calibrated(s, bestT_iter2), cap=0.60, max_flips=None)

## 3rd iteration:

In [ ]:
# ---- Cache namespaces ----
def set_cache_namespace(ns="dev"):
    """
    Switch the on-disk cache file used by chat_text_cached().
    Call set_cache_namespace("dev") during tuning; set_cache_namespace("test") before your final test run.
    """
    global CACHE_PATH
    CACHE_PATH = f"llm_cache_{ns}.db"
    print("Using cache:", CACHE_PATH)

In [ ]:
# ---- Retrieval pool & TF-IDF (reusable) ----
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

try:
    ALLOWED
except NameError:
    ALLOWED = ["happiness","sadness","anger","surprise","fear","disgust","neutral"]

# Survivor-domain contrastive anchors (short, factual, in-style with your data)
CONTRASTIVE_CORE = [
    # happiness
    ("Good job, Sam.", "happiness"),
    ("I'm proud of myself.", "happiness"),

    # sadness
    ("I'm devastated.", "sadness"),
    ("It breaks my heart.", "sadness"),

    # anger
    ("That's unfair.", "anger"),
    ("God damn it!", "anger"),

    # surprise (unexpectedness phrasing common in transcripts)
    ("I can't believe she spilled the beans.", "surprise"),
    ("All of a sudden, my name's getting thrown around.", "surprise"),

    # fear
    ("I'm worried they'll target me.", "fear"),
    ("It's pretty scary.", "fear"),

    # disgust
    ("That's disgusting.", "disgust"),
    ("It's making me feel sick.", "disgust"),

    # neutral (host narration / mechanics)
    ("Survivors ready?", "neutral"),
    ("I'll read the votes.", "neutral"),
    ("Sam wins individual immunity.", "neutral"),
    ("The tribe has spoken.", "neutral"),
    ("Six are left.", "neutral"),
]

def build_pool(df, per_class_caps=None):
    """
    Build a capped retrieval pool per label from df.
    Caps control class exposure in retrieval (prevents overfitting to neutrals).
    """
    if per_class_caps is None:
        per_class_caps = {
            "neutral": 80, "happiness": 60, "fear": 60, "sadness": 60,
            "anger": 50, "surprise": 50, "disgust": 20
        }
    frames = []
    for lab in ALLOWED:
        cap = per_class_caps.get(lab, 50)
        frames.append(df[df["Emotion"] == lab].head(cap))
    pool = pd.concat(frames).reset_index(drop=True)
    return pool

# Build initial pool from your current dev split
pool_iter3 = build_pool(df_dev)

# TF-IDF index for retrieval
tfidf_iter3 = TfidfVectorizer(max_features=20000, ngram_range=(1,2), lowercase=True, min_df=1)
X_pool_iter3 = tfidf_iter3.fit_transform(pool_iter3["Sentence"].tolist())

def retrieve_examples_iter3(query_text, k=6, per_label_max=2):
    """
    Pure retrieval from dev pool using TF-IDF cosine similarity,
    with a small per-label cap to avoid single-class flooding.
    """
    if not isinstance(query_text, str) or not query_text.strip():
        return []
    qv = tfidf_iter3.transform([query_text])
    sims = cosine_similarity(qv, X_pool_iter3)[0]
    idx = np.argsort(-sims)
    shots, per_lab = [], {lab: 0 for lab in ALLOWED}
    for i in idx:
        row = pool_iter3.iloc[i]
        lab = row["Emotion"]
        if per_lab[lab] < per_label_max:
            candidate = (row["Sentence"], lab)
            # de-dup against already added
            if candidate not in shots:
                shots.append(candidate)
                per_lab[lab] += 1
        if len(shots) >= k:
            break
    # ensure at least one neutral anchor
    if not any(l == "neutral" for _, l in shots):
        neut = pool_iter3[pool_iter3["Emotion"] == "neutral"]
        if not neut.empty:
            candidate = (neut.iloc[0]["Sentence"], "neutral")
            if candidate not in shots and len(shots) < k:
                shots.append(candidate)
    return shots[:k]

def retrieve_examples_mixed_iter3(text, k=6, per_label_max=2, mix=0.5):
    """
    Mixed few-shots:
      - Top-N retrieved from TF-IDF pool (episode-style)
      - Fill the rest with domain-true contrastive anchors (Survivor phrasing)
    This reduces overfitting to dev phrasing and stabilizes test performance.
    """
    # how many from retrieval vs anchors
    k_retr = max(2, int(round(k * mix)))
    k_anchor = max(0, k - k_retr)

    rshots = retrieve_examples_iter3(text, k=k_retr, per_label_max=per_label_max)

    # add anchors without duplicating existing pairs
    anchors = []
    seen = set(rshots)
    for s, l in CONTRASTIVE_CORE:
        if (s, l) not in seen:
            anchors.append((s, l))
            seen.add((s, l))
        if len(anchors) >= k_anchor:
            break

    shots = (rshots + anchors)[:k]

    # final safety: ensure at least one neutral
    if not any(l == "neutral" for _, l in shots):
        # try to inject a neutral anchor from CONTRASTIVE_CORE
        for s, l in CONTRASTIVE_CORE:
            if l == "neutral" and (s, l) not in shots:
                shots = (shots + [(s, l)])[:k]
                break

    return shots

def refresh_retriever_iter3(df_for_pool):
    """
    Rebuild the retrieval pool and TF-IDF index (use after changing your split).
    """
    global pool_iter3, tfidf_iter3, X_pool_iter3
    pool_iter3 = build_pool(df_for_pool)
    tfidf_iter3 = TfidfVectorizer(max_features=20000, ngram_range=(1,2), lowercase=True, min_df=1)
    X_pool_iter3 = tfidf_iter3.fit_transform(pool_iter3["Sentence"].tolist())
    print("Retriever refreshed:", pool_iter3.shape, "sentences")

In [ ]:
# ---- Mixed few-shots: retrieved + contrastive archetypes ----
# Survivor-domain contrastive anchors (short, factual, in-style with our data)
CONTRASTIVE_CORE = [
    # happiness
    ("Good job, Sam.", "happiness"),
    ("I'm proud of myself.", "happiness"),

    # sadness
    ("I'm devastated.", "sadness"),
    ("It breaks my heart.", "sadness"),

    # anger
    ("That's unfair.", "anger"),
    ("God damn it.", "anger"),

    # surprise (unexpectedness phrasing common in transcripts)
    ("I can't believe she spilled the beans.", "surprise"),
    ("All of a sudden, my name's getting thrown around.", "surprise"),

    # fear
    ("I'm worried they'll target me.", "fear"),
    ("It's pretty scary.", "fear"),

    # disgust
    ("That's disgusting.", "disgust"),
    ("It's making me feel sick.", "disgust"),

    # neutral (host narration / mechanics)
    ("Survivors ready?", "neutral"),
    ("I'll read the votes.", "neutral"),
    ("Sam wins individual immunity.", "neutral"),
    ("The tribe has spoken.", "neutral"),
    ("Six are left.", "neutral"),
]

def retrieve_examples_mixed_iter3(text, k=6, per_label_max=2, mix=0.5):
    """
    Mixed few-shots:
      - Top-N retrieved from TF-IDF pool (episode-style)
      - Fill the rest with domain-true contrastive anchors (Survivor phrasing)
    This reduces overfitting to dev phrasing and stabilizes test performance.
    """
    # how many from retrieval vs anchors
    k_retr = max(2, int(round(k * mix)))
    k_anchor = max(0, k - k_retr)

    rshots = retrieve_examples_iter3(text, k=k_retr, per_label_max=per_label_max)

    # add anchors without duplicating existing pairs
    anchors = []
    seen = set(rshots)
    for s, l in CONTRASTIVE_CORE:
        if (s, l) not in seen:
            anchors.append((s, l))
            seen.add((s, l))
        if len(anchors) >= k_anchor:
            break

    shots = (rshots + anchors)[:k]

    # final safety: ensure at least one neutral
    if not any(l == "neutral" for _, l in shots):
        for s, l in CONTRASTIVE_CORE:
            if l == "neutral" and (s, l) not in shots:
                shots = (shots + [(s, l)])[:k]
                break

    return shots

In [ ]:
# ---- Prompts (domain-aligned) ----
DEFS3 = """Definitions (Survivor domain):
- happiness: positive affect/relief/success (e.g., "I'm proud of myself.", "Good job.").
- sadness: loss/sorrow/regret (e.g., "I'm devastated.", "It breaks my heart.").
- anger: irritation/hostility/injustice (e.g., "God damn it.", "That's unfair.").
- surprise: unexpectedness/shock (e.g., "I can't believe she spilled the beans.", "All of a sudden my name's thrown around.").
- fear: threat/anxiety/worry (e.g., "I'm worried they'll target me.", "It's pretty scary.").
- disgust: aversion/moral revulsion (e.g., "That's disgusting.", "It's making me feel sick.").
- neutral: host narration, game mechanics, or backchannel/fillers:
  Examples: "Previously on...", "Survivors ready?", "Go.", "Come on in guys.", "I'll read the votes.",
  "The tribe has spoken.", "Sam wins individual immunity.", "Grab your torches.", "Six are left.",
  and strategy-only lines with no explicit affect: "We need a plan.", "We're voting Michelle.", "Flip or draw rocks." """

RULES3 = """Rules:
- Label 'neutral' for host narration, challenge descriptions, vote reading, and purely strategic statements without explicit affect.
- A question mark does NOT imply surprise. Use 'surprise' only for explicit unexpectedness ("I can't believe…", "didn't expect…").
- Choose exactly one label from [happiness, sadness, anger, surprise, fear, disgust, neutral]."""

def fewshot_block3(pairs):
    return "Few-shot (Sentence -> Label):\n" + "\n".join([f"\"{s}\" -> {l}" for s,l in pairs])

def json_prompt_iter3(text, shots):
    return f"""{DEFS3}

{RULES3}

{fewshot_block3(shots)}

Respond ONLY with strict JSON:
{{"label":"<one_of_allowed>", "probs":{{"happiness":x,"sadness":x,"anger":x,"surprise":x,"fear":x,"disgust":x,"neutral":x}}}}
Probabilities must be non-negative and sum to 1.

Sentence: "{text}"
"""

def no_neutral_prompt_iter3(text, shots):
    return f"""{DEFS3}

Rules (no-neutral):
- Choose only from [happiness, sadness, anger, surprise, fear, disgust]. Do NOT output 'neutral'.

{fewshot_block3(shots)}

Answer with ONE label only.

Sentence: "{text}"
"""

def ask_json_iter3(text, shots, temperature=0.0, max_tokens=220):
    raw = chat_text_cached(json_prompt_iter3(text, shots), temperature=temperature, max_tokens=max_tokens)
    lab, probs = parse_json_probs(raw)
    return lab, probs, raw

def ask_label_no_neutral_iter3(text, shots, temperature=0.2, max_tokens=16):
    raw = chat_text_cached(no_neutral_prompt_iter3(text, shots), temperature=temperature, max_tokens=max_tokens)
    lab = parse_label_plain(raw)
    return lab, raw

In [ ]:
import re

# ---- Priors & aggregation (neutral-friendly) ----
def class_priors_from_dev_iter3(df, neutral_soften=1.00, power=0.5):
    vc = df["Emotion"].value_counts().reindex(ALLOWED).fillna(1)
    inv = 1.0 / np.power(vc.values, power)
    pri = inv / inv.sum()
    pri = dict(zip(ALLOWED, pri))
    pri["neutral"] *= neutral_soften  # keep 1.00 to avoid anti-neutral bias
    s = sum(pri.values())
    for k in pri: pri[k] /= s
    return pri

priors_iter3 = class_priors_from_dev_iter3(df_dev, neutral_soften=1.00, power=0.5)

def combine_probs_with_priors_iter3(prob_list, priors, gamma=0.55, neutral_demotion=1.00):
    agg = {k:0.0 for k in ALLOWED}
    for probs in prob_list:
        for lab in ALLOWED:
            agg[lab] += (1-gamma)*probs.get(lab,0.0) + gamma*priors.get(lab, 1.0/len(ALLOWED))
    agg["neutral"] *= float(neutral_demotion)  # 1.00 = no demotion
    s = sum(agg.values()) or 1.0
    agg = {k: v/s for k,v in agg.items()}
    label = max(agg, key=agg.get)
    return label, agg

# ---- Happiness vs Surprise resolver ----
def happy_surprise_prompt_iter3(text):
    return f"""Choose ONLY 'happiness' or 'surprise'.
Unexpectedness/shock → 'surprise'. Achievement/joy/relief → 'happiness'.
Answer with one word.
Sentence: "{text}"
"""

def resolve_happy_surprise_iter3(text):
    raw = chat_text_cached(happy_surprise_prompt_iter3(text), temperature=0.0, max_tokens=8)
    lab = parse_label_plain(raw)
    return lab if lab in ("happiness","surprise") else None

# ---- Domain regexes for neutral & affect cues ----
HOST_NEUTRAL_RE = re.compile(
    r"\b(previously on|survivors ready\??|come on in guys|go!?|i(?:'|’)ll read the votes|the tribe has spoken|grab your torches|"
    r"immunity (?:is )?back up for grabs|wins (?:individual )?immunity|draw for rocks?|time to vote|i(?:'|’)ll count you in|three, two, one|"
    r"member of our jury|person voted out|bring me (?:your|a) torch|next time on australian survivor)\b",
    re.I
)

STRATEGY_NEUTRAL_RE = re.compile(
    r"\b(plan|flip|rocks?|alliance|vote|votes|tribal council|immunity necklace|resume)\b", re.I
)

AFFECT_RE = re.compile(
    r"\b(i(?:'| )?m|i am|i feel|it (?:scares|makes) me|i was)\s+"
    r"(?:devastated|sad|gutted|angry|furious|mad|worried|afraid|scared|nervous|anxious|proud|happy|"
    r"thrilled|shocked|surprised|disgusted|sick|upset)\b",
    re.I
)

# Survivor-style surprise cues (no exclamation needed)
SURPRISE_CUE_RE = re.compile(
    r"\b(i can'?t believe|all of a sudden|didn'?t expect|what (?:just )?happened|that (?:was|is) a (?:twist|shock)|wow)\b",
    re.I
)

def has_explicit_surprise(text: str) -> bool:
    return bool(SURPRISE_CUE_RE.search(text or ""))

# Fallbacks if earlier globals aren't defined
try:
    FILLER_RE
except NameError:
    FILLER_RE = re.compile(r'^(yeah|yep|ok(ay)?|mm+|uh+|oh)\.?$', re.I)

try:
    LEX
except NameError:
    LEX = {
        "disgust":   re.compile(r"\b(disgust(ing)?|gross|feel sick|nausea|yuck|ew+w?)\b", re.I),
        "happiness": re.compile(r"\b(thanks|thank you|glad|happy|proud|win|victor(y|ies)|congrats?)\b", re.I),
        "sadness":   re.compile(r"\b(breaks my heart|farewell|gutted|sad|goodbye|loss|heartbroken|devastated)\b", re.I),
        "anger":     re.compile(r"\b(god ?damn|wrong one|angry|furious|frustrat(ed|ing)|mad)\b|^go!?$", re.I),
        "surprise":  re.compile(r"\b(didn'?t expect|what just happened|twist|shocked|wow|no way|ready\?)\b", re.I),
        "fear":      re.compile(r"\b(dangerous|target me|worried|afraid|scared|terrified|threat|unsafe|nervous|anxious)\b", re.I),
    }

def has_explicit_affect(t: str) -> bool:
    return bool(AFFECT_RE.search(t or ""))

# override "strong cue" to avoid '?' bias
def has_strong_emotion_cue(text):
    t = text or ""
    if "!" in t:
        return True
    if has_explicit_affect(t):
        return True
    return any(rx.search(t) for rx in LEX.values())

def rule_label_domain_neutral(text):
    t = (text or "").strip().lower()
    if not t or len(t) < 3:
        return "neutral"
    if FILLER_RE.match(t):
        return "neutral"
    if HOST_NEUTRAL_RE.search(t):
        return "neutral"
    # strategy without explicit affect → neutral
    if STRATEGY_NEUTRAL_RE.search(t) and not has_explicit_affect(t):
        return "neutral"
    return None

# Seamless wrapper: prepend domain-neutral rule to any existing rule_label()
try:
    _prior_rule_label = rule_label
except NameError:
    _prior_rule_label = lambda x: None

def rule_label(text):
    return rule_label_domain_neutral(text) or _prior_rule_label(text)

# ---- Post-nudge (with optional surprise rescue) ----
def post_nudge_generalize_iter3(label, text):
    t = (text or "").lower()
    if HOST_NEUTRAL_RE.search(t):
        return "neutral"  # narration/mechanics
    # Optional rescue: if explicit surprise cues present but model says neutral
    if label == "neutral" and has_explicit_surprise(t):
        return "surprise"
    if label == "happiness" and any(k in t for k in ["thanks","glad","proud","we did it","so happy"]):
        return "happiness"
    return label

def post_nudge_generalize_iter3(label, text):
    t = (text or "").lower()
    if HOST_NEUTRAL_RE.search(t):
        return "neutral"
    if label == "neutral" and has_explicit_surprise(text):  # <-- optional rescue
        return "surprise"
    if label == "happiness" and any(k in t for k in ["thanks","glad","proud","we did it","so happy"]):
        return "happiness"
    return label


In [ ]:
# ---- Iter 3 classifier (generalization-focused) ----
def classify_combo_iter3_generalize(
    text,
    t_stage1=0.0,
    t_stage2_list=(0.0, 0.2, 0.35),   # small diversity only when needed
    low_conf_thresh=0.55,             # was 0.40
    prior_gamma=0.55,                 # milder priors
    neutral_demotion=1.00             # not used below; we pass 0.90 at combine
):
    # A) High-precision lexical rules first (now includes domain-neutral wrapper)
    rb = rule_label(text)
    if rb:
        return rb

    # B) Mixed few-shots (retrieved + domain anchors)
    shots = retrieve_examples_mixed_iter3(text, k=6, per_label_max=2, mix=0.5)

    # C) Stage 1: JSON probs
    lab1, p1, _ = ask_json_iter3(text, shots, temperature=t_stage1, max_tokens=220)
    prob_list = [p1]

    # D) Escalate only when justified
    explicit_affect = has_explicit_affect(text)
    explicit_surprise = has_explicit_surprise(text)   # <-- NEW
    top_p = p1.get(lab1, 0.0)
    escalate = ((lab1 == "neutral") and (explicit_affect or explicit_surprise)) or \
               ((top_p < low_conf_thresh) and (lab1 != "neutral"))

    if escalate and t_stage2_list:
        for t in t_stage2_list:
            lab2, _ = ask_label_no_neutral_iter3(text, shots, temperature=t, max_tokens=12)
            if lab2 in ("happiness","sadness","anger","surprise","fear","disgust"):
                peaked = {k: 1e-6 for k in ALLOWED}; peaked[lab2] = 1.0
                prob_list.append(peaked)

    # E) H↔S resolver only when both high-ish and close, and not host/narration
    if (not HOST_NEUTRAL_RE.search(text or "")
        and p1.get("happiness",0) > 0.25 and p1.get("surprise",0) > 0.25
        and abs(p1.get("happiness",0) - p1.get("surprise",0)) < 0.15):
        hs = resolve_happy_surprise_iter3(text)
        if hs:
            peaked = {k: 1e-6 for k in ALLOWED}; peaked[hs] = 1.0
            prob_list.append(peaked)

    # F) Aggregate with a gentle anti-neutral (helps test where neutral is overpredicted)
    final_label, _ = combine_probs_with_priors_iter3(
        prob_list, priors_iter3,
        gamma=prior_gamma,
        neutral_demotion=0.90   # <-- your change kept
    )
    return post_nudge_generalize_iter3(final_label, text)

In [ ]:
# ---- Optional calibration (reuse bestT from Iter-2 if you have it) ----
BEST_T = globals().get("bestT_iter2", 1.0)

def temperature_scale_iter3(probs: dict, T: float) -> dict:
    eps = 1e-8
    q = {k: (probs[k]+eps)**(1.0/T) for k in probs}
    s = sum(q.values()) or 1.0
    return {k: v/s for k,v in q.items()}

def classify_combo_iter3_calibrated(text, T=BEST_T):
    # A) Rules first
    rb = rule_label(text)
    if rb: 
        return rb

    # B) Mixed few-shots
    shots = retrieve_examples_mixed_iter3(text, k=6, per_label_max=2, mix=0.5)

    # C) Stage 1
    lab1, p1, _ = ask_json_iter3(text, shots, temperature=0.0, max_tokens=220)
    prob_list = [p1]

    # D) Escalation (same stricter logic as generalize)
    explicit_affect = has_explicit_affect(text)
    explicit_surprise = has_explicit_surprise(text)
    top_p = p1.get(lab1, 0.0)
    escalate = ((lab1 == "neutral") and (explicit_affect or explicit_surprise)) or (top_p < low_conf_thresh and lab1 != "neutral")

    # E) H↔S resolver (guard against host/narration)
    if (not HOST_NEUTRAL_RE.search(text or "")
        and p1.get("happiness",0) > 0.25 and p1.get("surprise",0) > 0.25
        and abs(p1.get("happiness",0) - p1.get("surprise",0)) < 0.15):
        hs = resolve_happy_surprise_iter3(text)
        if hs:
            peaked = {k: 1e-6 for k in ALLOWED}; peaked[hs] = 1.0
            prob_list.append(peaked)

    # F) Aggregate, then temperature-scale before argmax
    _, agg = combine_probs_with_priors_iter3(prob_list, priors_iter3, gamma=0.55, neutral_demotion=1.00)
    q = temperature_scale_iter3(agg, T)
    return post_nudge_generalize_iter3(max(q, key=q.get), text)

In [ ]:
# ---- Evaluate with cache hygiene ----
# 1) DEV (tuning)
set_cache_namespace("dev")
_ = evaluate("Iter3 — Generalize (no calib)", df_dev, classify_combo_iter3_generalize, show_cm=True)
_ = evaluate("Iter3 — Generalize + calib", df_dev, classify_combo_iter3_calibrated, show_cm=True)

# 2) TEST (fresh cache, unbiased)
set_cache_namespace("test")
_ = evaluate("Iter3 — Generalize + calib (TEST)", df_test, classify_combo_iter3_calibrated, show_cm=True)

In [ ]:
# --- Run this once after you've loaded df_raw and created df (Sentence, Emotion) ---

# 0) Ensure chronological order
if "Start Time" in df_raw.columns:
    df_raw = df_raw.sort_values("Start Time", kind="mergesort").reset_index(drop=True)
    df = df_raw.rename(columns={"Emotion_core":"Emotion"})[["Sentence","Emotion"]].copy()
    df["Sentence"] = df["Sentence"].astype(str).str.strip()
    df["Emotion"]  = df["Emotion"].astype(str).str.lower().str.strip()

# 1) Quintile stats (5 x 20% chronological chunks)
import numpy as np
import pandas as pd

ALLOWED = ["happiness","sadness","anger","surprise","fear","disgust","neutral"]

def make_quintiles(df):
    n = len(df)
    bounds = [(int(i*n/5), int((i+1)*n/5)) for i in range(5)]
    return bounds  # [(start,end), ...]

def chunk_stats(df, label_col="Emotion"):
    bounds = make_quintiles(df)
    overall = df[label_col].value_counts().reindex(ALLOWED, fill_value=0)
    rows = []
    for i,(a,b) in enumerate(bounds, start=1):
        part = df.iloc[a:b]
        vc = part[label_col].value_counts().reindex(ALLOWED, fill_value=0)
        rate = (vc / max(len(part),1)).round(3)
        rows.append({
            "chunk": i,
            "rows": len(part),
            **{f"cnt_{lab}": int(vc[lab]) for lab in ALLOWED},
            **{f"rate_{lab}": float(rate[lab]) for lab in ALLOWED},
            "idx_from": a, "idx_to": b-1
        })
    return pd.DataFrame(rows).set_index("chunk"), overall

stats_df, overall_counts = chunk_stats(df)
display(stats_df)

# 2) Which 20% has the most 'disgust'?
most_disgust_chunk = stats_df["cnt_disgust"].astype(int).idxmax()
most_disgust_count = int(stats_df.loc[most_disgust_chunk, "cnt_disgust"])
print(f"\nChunk with MOST 'disgust': chunk {most_disgust_chunk} "
      f"({stats_df.loc[most_disgust_chunk,'idx_from']}–{stats_df.loc[most_disgust_chunk,'idx_to']}), "
      f"count={most_disgust_count}")

# 3) (Optional) Pick a 20% test chunk that avoids disgust spikes and minimizes drift
def _dist(counts):
    arr = np.array([counts.get(l,0) for l in ALLOWED], dtype=float)
    s = arr.sum()
    return arr/s if s>0 else np.ones(len(ALLOWED))/len(ALLOWED)

def _chi_square(p, q, eps=1e-9):
    return float(((p-q)**2 / (q+eps)).sum())

overall_dist = _dist(overall_counts.to_dict())

candidates = []
for i in stats_df.index:
    cnts = {lab: int(stats_df.loc[i, f"cnt_{lab}"]) for lab in ALLOWED}
    p = _dist(cnts)
    chi = _chi_square(p, overall_dist)
    candidates.append({
        "chunk": i,
        "cnt_disgust": cnts["disgust"],
        "chi2_vs_overall": chi,
        "rows": int(stats_df.loc[i, "rows"]),
        "idx_from": int(stats_df.loc[i, "idx_from"]),
        "idx_to": int(stats_df.loc[i, "idx_to"]),
    })

cand_df = pd.DataFrame(candidates).set_index("chunk").sort_values(["cnt_disgust","chi2_vs_overall"])
print("\nCandidates (sorted by lowest disgust, then smallest drift):")
display(cand_df)

best_chunk = cand_df.index[0]
print(f"\nRecommended 20% TEST chunk (avoids disgust spike, lowest drift): chunk {best_chunk} "
      f"({cand_df.loc[best_chunk,'idx_from']}–{cand_df.loc[best_chunk,'idx_to']})")

# 4) If you want to USE that recommended chunk as TEST and rebuild:
a = int(stats_df.loc[best_chunk, "idx_from"])
b = int(stats_df.loc[best_chunk, "idx_to"]) + 1
df_test_quint = df.iloc[a:b].reset_index(drop=True)
df_dev_quint  = pd.concat([df.iloc[:a], df.iloc[b:]]).reset_index(drop=True)

print(f"\nDEV size: {len(df_dev_quint)} | TEST size: {len(df_test_quint)}")

# Rebuild retriever + priors for this split (no leakage)
refresh_retriever_iter3(df_dev_quint)
priors_iter3 = class_priors_from_dev_iter3(df_dev_quint, neutral_soften=1.00, power=0.5)

# Evaluate cleanly with fresh caches
set_cache_namespace("dev_quint")
_ = evaluate("Iter3 — Generalize+calib (DEV quint)", df_dev_quint, classify_combo_iter3_calibrated, show_cm=True)

set_cache_namespace("test_quint")
_ = evaluate("Iter3 — Generalize+calib (TEST quint)", df_test_quint, classify_combo_iter3_calibrated, show_cm=True)